# Phase 6: Train U-Net Defect Segmentation Model in Google Colab (Tesla T4 GPU)

This notebook trains an encoder-decoder **U-Net with skip connections** and **Combined BCE + Soft Dice Loss** for pixel-level boundary localization on manufacturing defects.

### Pipeline Highlights:
1. **GPU Gatekeeper**: Verifies Tesla T4 allocation and CUDA readiness.
2. **Dual Augmentation**: Synchronized spatial rotations and flips locked between image and mask; photometric jitter isolated to image.
3. **Combined Loss**: Balances global background gradients (BCE) with severe defect class imbalance resilience (Soft Dice).
4. **Evaluation**: Computes Mean IoU (Jaccard), Dice (F1), and Pixel Accuracy on hold-out test set.
5. **Artifact Publishing**: Exports best PyTorch checkpoint and uploads to Hugging Face Hub.

In [ ]:
# 1. Environment & Dependencies
!pip install -q huggingface_hub matplotlib opencv-python

In [ ]:
# 2. GPU Verification
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected! Please enable T4 GPU in Runtime -> Change runtime type.")

print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# 3. Hugging Face Login
import os
from huggingface_hub import HfApi, login

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token)
    api = HfApi()
    print(f"Logged in as: {api.whoami()['name']}")
else:
    print("HF_TOKEN not found. Artifact upload will be skipped.")

In [ ]:
# 4. Launch Training Script
!python training/colab/segmentation/06_train_unet.py